[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E5%88%9D%E8%A6%8Bpandas/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E5%8F%B0%E7%81%A3%E4%BA%BA%E6%9C%80%E6%84%9B%E7%9C%8B%E4%BB%80%E9%BA%BC%E9%9B%BB%E5%BD%B1_%E7%AC%AC%E4%B8%80%E4%BB%BDpandas%E5%A0%B1%E8%A1%A8.ipynb)

# 🎬 台灣人最愛看什麼電影？用 pandas 做第一份票房報表

> 🎯 **情境**：你在影評社團負責「年度回顧」專題。文化部公開了**全國每一部電影的票房**——從 2016 年到現在，7,000 多部電影。用 Python 的清單和字典處理這麼多資料太辛苦了，今天換上新武器 **pandas**：它會把資料變成一張「會說話的 Excel 表」，讓你一行程式就回答「**2025 年最賣座的是誰？國片表現如何？**」

| 素材檔案 | 內容 |
| --- | --- |
| `全國電影票房_2016至今.json` | 文化部全國電影票房：2016 年至今 7,000 多部電影的國別、上映日、戲院數、票數與票房金額（JSON） |

| 📡 資料來源（政府資料開放平台） | 提供機關 | 下載日期 |
| --- | --- | --- |
| [全國電影票房統計數據](https://data.gov.tw/dataset/94224) | 文化部 | 2026-10-24 |

> ✅ 這些都是**真實的政府開放資料**，依「政府資料開放授權條款－第1版」使用。課堂上統一使用上表日期的快照，所以每個人跑出來的數字都一樣。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | Series——一欄有「名牌」的數字 |
| ⬜ 任務 2 | DataFrame——把 7,000 多部電影變成一張表 |
| ⬜ 任務 3 | 新增欄位——讓表格回答更多問題 |
| ⬜ 任務 4 | 篩選——2025 年的國片表現如何？ |
| ⬜ 任務 5 | 索引對齊——2024 年和 2025 年，哪些國家的電影變多了？ |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「老闆想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '全國電影票房_2016至今.json',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E5%88%9D%E8%A6%8Bpandas/%E5%88%9D%E8%A6%8Bpandas_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['全國電影票房_2016至今.json']


## 任務 1：Series——一欄有「名牌」的數字

**白話說明**：`Series` 就像 Excel 的**一欄**，但每個值都掛著一個名牌（index）。用名牌取值，比記「第幾個」直覺多了。

In [2]:
import pandas as pd

top5 = pd.Series([8.57, 7.75, 7.57, 5.74, 4.93],                 # 2025 年前 5 名的票房（億元）
                 index=['鬼滅之刃 無限城篇', '陽光女子合唱團', '動物方城市2', 'F1電影', '阿凡達：火與燼'],
                 name='票房(億)')
display(top5)
print('陽光女子合唱團：', top5['陽光女子合唱團'], '億')          # 用名牌取值
print('前 5 名合計：', top5.sum().round(2), '億；平均：', top5.mean().round(2), '億')
print('破 6 億的：', top5[top5 > 6].index.tolist())              # 條件篩選

鬼滅之刃 無限城篇    8.57
陽光女子合唱團      7.75
動物方城市2       7.57
F1電影         5.74
阿凡達：火與燼      4.93
Name: 票房(億), dtype: float64

陽光女子合唱團： 7.75 億
前 5 名合計： 34.56 億；平均： 6.91 億
破 6 億的： ['鬼滅之刃 無限城篇', '陽光女子合唱團', '動物方城市2']


## 任務 2：DataFrame——把 7,000 多部電影變成一張表

**白話說明**：`DataFrame` 就是一整張表格：每一欄都是一個 Series。JSON 裡的電影清單是「一部電影一個字典」，直接丟給 `pd.DataFrame()` 就變成表格。

In [3]:
import json

with open('全國電影票房_2016至今.json', encoding='utf-8-sig') as f:
    data = json.load(f)

movies = pd.DataFrame(data['List'])                                # 字典的清單 → 表格（一個字典 = 一列）
print('幾列 × 幾欄：', movies.shape)
movies.head()

幾列 × 幾欄： (7328, 11)


,Country,Name,ReleaseDate,Issue,Produce,TheaterCount,Tickets,TicketChangeRate,Amounts,TotalTickets,TotalAmounts
0,日本,《我在學校旅行中被分到了一個糟糕的小組》高校同學會－大銀幕限定青春重映,2026-10-11T00:00:00,秀泰全球影城股份有限公司,NaN,1,94,0,122200.0,94,122200.0
1,日本,YORUSHIKA LIVE「盜作」,2026-10-09T00:00:00,薩摩亞商羚邦(亞洲)有限公司台灣分公司,NaN,11,15023,0,7394950.0,15023,7394950.0
2,美國,金劫任務,2026-10-08T00:00:00,車庫娛樂股份有限公司,NaN,72,17535,0,4353605.0,17535,4353605.0
3,越南,鬼入室,2026-10-08T00:00:00,暗光鳥娛樂有限公司,NaN,44,2664,0,695375.0,2664,695375.0
4,越南,驚天劫機,2026-10-08T00:00:00,香港商甲上娛樂有限公司台灣分公司,NaN,30,4827,0,1275241.0,4827,1275241.0


In [4]:
movies.describe()                                                  # 數字欄位的統計摘要

,TheaterCount,Tickets,TicketChangeRate,Amounts,TotalTickets,TotalAmounts
count,7328.000000,7.328000e+03,7328.0,7.328000e+03,7.328000e+03,7.328000e+03
mean,29.602211,4.493815e+04,0.0,1.127989e+07,4.493815e+04,1.127989e+07
std,30.868824,1.877451e+05,0.0,4.870668e+07,1.877451e+05,4.870668e+07
min,1.000000,-1.000000e+00,0.0,-1.000000e+00,-1.000000e+00,-1.000000e+00
25%,6.000000,4.885000e+02,0.0,1.150688e+05,4.885000e+02,1.150688e+05
50%,16.000000,2.206000e+03,0.0,4.997375e+05,2.206000e+03,4.997375e+05
75%,47.000000,1.165775e+04,0.0,2.785401e+06,1.165775e+04,2.785401e+06
max,125.000000,3.492311e+06,0.0,9.104561e+08,3.492311e+06,9.104561e+08


> 🕵️ **仔細看 `describe()` 的結果**，你會發現兩個「真實資料的怪現象」：
> 1. `TicketChangeRate`（票房變動率）**全部都是 0**——這一欄沒有任何資訊。
> 2. `Tickets` 和 `TotalTickets`、`Amounts` 和 `TotalAmounts` 的數字**一模一樣**——重複了。
> 3. 票數、票房的最小值（min）是 **-1**——票房不可能是負的！這是系統用 -1 代表「沒有資料」。
>
> 真實資料常常有沒用、重複或「假數字」的欄位，**先看懂、再處理**，算出來的結果才可信。

In [5]:
movies = movies.drop(columns=['TicketChangeRate', 'Tickets', 'Amounts', 'Produce'])   # 刪掉沒用、重複、幾乎空白的欄位
movies = movies.rename(columns={'Country': '國別', 'Name': '片名', 'ReleaseDate': '上映日', 'Issue': '發行商',
                                'TheaterCount': '戲院數', 'TotalTickets': '票數', 'TotalAmounts': '票房'})
print('票房是 -1（沒有資料）的電影：', (movies['票房'] < 0).sum(), '部')
movies = movies[movies['票房'] > 0]                                # 只留下有票房資料的電影
print('清理後：', movies.shape)
movies.head()

票房是 -1（沒有資料）的電影： 1 部
清理後： (7322, 7)


,國別,片名,上映日,發行商,戲院數,票數,票房
0,日本,《我在學校旅行中被分到了一個糟糕的小組》高校同學會－大銀幕限定青春重映,2026-10-11T00:00:00,秀泰全球影城股份有限公司,1,94,122200.0
1,日本,YORUSHIKA LIVE「盜作」,2026-10-09T00:00:00,薩摩亞商羚邦(亞洲)有限公司台灣分公司,11,15023,7394950.0
2,美國,金劫任務,2026-10-08T00:00:00,車庫娛樂股份有限公司,72,17535,4353605.0
3,越南,鬼入室,2026-10-08T00:00:00,暗光鳥娛樂有限公司,44,2664,695375.0
4,越南,驚天劫機,2026-10-08T00:00:00,香港商甲上娛樂有限公司台灣分公司,30,4827,1275241.0


## 任務 3：新增欄位——讓表格回答更多問題

**白話說明**：`表格['新欄位'] = 計算式`，整欄一次算完，不用寫迴圈。`.str[:4]` 取文字的前 4 個字。

In [6]:
movies['年'] = movies['上映日'].str[:4].astype(int)                # '2025-08-06T00:00:00' → 2025
movies['票房(億)'] = (movies['票房'] / 1e8).round(2)
movies['平均票價'] = (movies['票房'] / movies['票數']).round(0)   # 一張票平均多少錢

movies.sort_values('票房', ascending=False).head(10)[['片名', '國別', '年', '戲院數', '票數', '票房(億)', '平均票價']]

,片名,國別,年,戲院數,票數,票房(億),平均票價
4914,復仇者聯盟:終局之戰,美國,2019,109,3492311,9.10,261.0
788,劇場版「鬼滅之刃」無限城篇 第一章 猗窩座再襲,日本,2025,100,2662857,8.57,322.0
519,陽光女子合唱團,中華民國,2025,91,2826914,7.75,274.0
596,動物方城市2,美國,2025,97,2607452,7.57,290.0
2521,阿凡達：水之道,美國,2022,111,2247505,7.50,334.0
2885,捍衛戰士: 獨行俠,美國,2022,113,2699599,7.41,274.0
6417,玩命關頭8,美國,2017,109,2674179,6.51,243.0
5680,復仇者聯盟：無限之戰,美國,2018,105,2596803,6.42,247.0
3822,劇場版「鬼滅之刃」無限列車篇,日本,2020,102,2607077,6.36,244.0
5581,侏羅紀世界:殞落國度,美國,2018,110,2554833,6.22,243.0


> 💡 **看懂結果**：平均票價大多在 240～330 元之間——這就是你去電影院買一張票的價錢。前 10 名幾乎都在 **90 家以上的戲院**同時上映，大片的「排片優勢」非常明顯。

## 任務 4：篩選——2025 年的國片表現如何？

**白話說明**：`表格[條件]` 只留下符合條件的列；兩個條件用 `&`（而且）連接，**每個條件都要加括號**。

In [7]:
tw2025 = movies[(movies['國別'] == '中華民國') & (movies['年'] == 2025)]          # 國片 而且 2025 年
print('2025 年國片數量：', len(tw2025), '部')
print('2025 年國片總票房：', round(tw2025['票房'].sum() / 1e8, 1), '億')
tw2025.sort_values('票房', ascending=False).head(5)[['片名', '上映日', '戲院數', '票房(億)']]

2025 年國片數量： 75 部
2025 年國片總票房： 17.3 億


,片名,上映日,戲院數,票房(億)
519,陽光女子合唱團,2025-12-31T00:00:00,91,7.75
748,96分鐘,2025-09-05T00:00:00,96,2.07
770,角頭－鬥陣欸,2025-08-21T00:00:00,98,1.89
595,大濛,2025-11-27T00:00:00,98,1.11
678,泥娃娃,2025-10-09T00:00:00,91,1.01


> 💡 **看懂結果**：2025 年國片前兩名《陽光女子合唱團》、《96分鐘》都破 2 億，前者更衝到 7.75 億、擠進全年第 2 名，是近年國片最好的成績之一。
>
> 🤖 **換你問 AI**：「我有一個電影票房的 DataFrame，欄位有片名、國別、年、票房。請篩選出 2025 年的國片，依票房由高到低列出前 5 名。」

## 任務 5：索引對齊——2024 年和 2025 年，哪些國家的電影變多了？

**白話說明**：`value_counts()` 數出每個國別有幾部電影，結果是一個以「國別」當名牌的 Series。
兩個 Series 相減時，pandas 會**依照名牌自動對齊**，不是依照順序——只出現在一邊的名牌，結果會是 `NaN`（沒有資料）。

In [8]:
count2024 = movies.loc[movies['年'] == 2024, '國別'].value_counts()
count2025 = movies.loc[movies['年'] == 2025, '國別'].value_counts()

compare = pd.DataFrame({'2024年': count2024, '2025年': count2025})   # 依「國別」自動對齊
compare['變化'] = compare['2025年'] - compare['2024年']
compare.sort_values('2025年', ascending=False).head(12)

,2024年,2025年,變化
國別,,,
日本,142.0,151.0,9.0
美國,106.0,129.0,23.0
中華民國,62.0,75.0,13.0
韓國,61.0,68.0,7.0
法國,34.0,45.0,11.0
香港,24.0,34.0,10.0
中國大陸,12.0,27.0,15.0
英國,28.0,22.0,-6.0
泰國,24.0,17.0,-7.0


In [9]:
no_pair = compare[compare['變化'].isna()]                           # 哪些國家出現 NaN？
print('只在其中一年出現的國別：', len(no_pair), '個')
no_pair.head(8)

只在其中一年出現的國別： 43 個


,2024年,2025年,變化
國別,,,
中國,1.0,NaN,NaN
中國大陸/法國,NaN,3.0,NaN
中國大陸/香港,NaN,3.0,NaN
中華民國/香港,NaN,1.0,NaN
丹麥/英國,1.0,NaN,NaN
亞美尼亞,1.0,NaN,NaN
俄羅斯,2.0,NaN,NaN
保加利亞,1.0,NaN,NaN


> 💡 **看懂結果**：有些國家只在其中一年有電影上映，另一年沒有名牌，所以「變化」算不出來，變成 `NaN`。
> 這不是錯誤，而是 pandas 很誠實地告訴你「**這裡沒有資料**」——下一步要決定：把 NaN 當成 0？還是刪掉？（練習 3 會用到）
>
> 🕵️ **再仔細看國別**：同一個地方有兩種寫法（「台灣」和「中華民國」、「中國」和「中國大陸」），合拍片還寫成「中國大陸/法國」。這種「寫法不統一」是真實資料最常見的問題，第 8 章「資料整理」會教你怎麼統一。

<details><summary>🔄 想看最新資料？點我</summary>

資料每天、每月都在更新。想用今天的數字重跑一次，可以把這段程式貼到新的一格執行：

```python
import pandas as pd, urllib.request, json
raw = urllib.request.urlopen('https://boxofficetw.tfai.org.tw/OpenData/statistic/since2016').read()
latest_movies = pd.DataFrame(json.loads(raw.decode('utf-8-sig'))['List'])
latest_movies.sort_values('ReleaseDate', ascending=False).head()     # 最近上映的電影
```

> ⚠️ 開放資料的欄位偶爾會改名或調整，如果最新資料跑出錯誤，正好練習看懂錯誤訊息、對照欄位名稱修正程式。

</details>

---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [10]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：日本電影的平均票價是多少？

<details><summary>💡 卡住了？點我看提示</summary>

先用 `movies['國別'] == '日本'` 篩選，再取出「平均票價」那一欄，最後 `.mean()`。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
japan = movies[movies[___] == '日本']
japan_price = japan['平均票價'].___()
print('日本片平均票價：', round(japan_price), '元，共', len(japan), '部')

In [ ]:
# @title ✅ 執行這格，檢查練習 1
check('japan_price', movies.loc[movies['國別'] == '日本', '平均票價'].mean())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
japan = movies[movies['國別'] == '日本']
japan_price = japan['平均票價'].mean()
print('日本片平均票價：', round(japan_price), '元，共', len(japan), '部')
```

</details>

### ✍️ 練習 2：哪一部電影「平均每家戲院」賣最多錢？

<details><summary>💡 卡住了？點我看提示</summary>

新增一欄「每家戲院票房」= 票房 ÷ 戲院數；`idxmax()` 回傳最大值的列名牌，再用 `movies.loc[名牌, '片名']` 查片名。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
movies['每家戲院票房'] = movies['票房'] / movies[___]
best_idx = movies['每家戲院票房'].___()
best_movie = movies.loc[best_idx, '片名']
print('平均每家戲院最會賺：', best_movie, round(movies.loc[best_idx, '每家戲院票房'] / 1e4), '萬元')

In [ ]:
# @title ✅ 執行這格，檢查練習 2
check('best_movie', movies.loc[(movies['票房'] / movies['戲院數']).idxmax(), '片名'])

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
movies['每家戲院票房'] = movies['票房'] / movies['戲院數']
best_idx = movies['每家戲院票房'].idxmax()
best_movie = movies.loc[best_idx, '片名']
print('平均每家戲院最會賺：', best_movie, round(movies.loc[best_idx, '每家戲院票房'] / 1e4), '萬元')
```

</details>

### ✍️ 練習 3：把 NaN 補成 0，重新計算「變化」，找出 2025 年電影變最多的國家

<details><summary>💡 卡住了？點我看提示</summary>

某國在某一年沒有電影，代表「0 部」，所以可以用 `fillna(0)` 補 0。補好之後再算一次「變化」，用 `idxmax()` 找最大。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
compare = compare.___(0)                           # 沒有電影 = 0 部
compare['變化'] = compare['2025年'] - compare['2024年']
most_up = compare['變化'].idxmax()
print('2025 年電影數增加最多的國家：', most_up, '，多了', int(compare.loc[most_up, '變化']), '部')

In [ ]:
# @title ✅ 執行這格，檢查練習 3
_c = pd.DataFrame({'2024年': movies.loc[movies['年'] == 2024, '國別'].value_counts(),
                   '2025年': movies.loc[movies['年'] == 2025, '國別'].value_counts()}).fillna(0)
check('most_up', (_c['2025年'] - _c['2024年']).idxmax())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
compare = compare.fillna(0)
compare['變化'] = compare['2025年'] - compare['2024年']
most_up = compare['變化'].idxmax()
print('2025 年電影數增加最多的國家：', most_up, '，多了', int(compare.loc[most_up, '變化']), '部')
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：Series——一欄有「名牌」的數字
- ✅ 任務 2：DataFrame——把 7,000 多部電影變成一張表
- ✅ 任務 3：新增欄位——讓表格回答更多問題
- ✅ 任務 4：篩選——2025 年的國片表現如何？
- ✅ 任務 5：索引對齊——2024 年和 2025 年，哪些國家的電影變多了？

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以交給老闆的報告。